In [ ]:
# SPR 2026 - BERTimbau MAX_LEN=512 + TTA (Test-Time Augmentation de Texto)
# Estrategia: Rodar inferencia 3x com variacoes de pre-processamento do texto
# e fazer media das probabilidades. Isso funciona como um ensemble sem retreino.
#
# Variacoes:
#   1. Texto estruturado (build_input_text) -- padrao do winner
#   2. Texto raw (clean_text apenas) -- sem extracao de secoes
#   3. Texto estruturado invertido (Achados primeiro, Indicacao depois)
#
# Hipotese: cada pre-processamento enfatiza partes diferentes do relatorio.
# A media suaviza erros de qualquer variacao individual.

import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# ==========================================================================
# CONFIG
# ==========================================================================
MAX_LEN     = 512
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v4/advanced_outputs_kaggle_4')
CONFIG_KEY   = 'bertimbau_large__cb_focal'
weights_dir  = WEIGHTS_BASE / 'weights' / CONFIG_KEY

# Calibracao do winner
BEST_TEMP       = 0.9580
BEST_THRESHOLDS = [0.9500, 1.6000, 1.3500, 1.0, 0.4000, 1.1500, 0.1]

print(f'Device: {DEVICE}')
print(f'Modo: TTA com 3 variacoes de texto')

In [ ]:
# ==========================================================================
# DATASET
# ==========================================================================
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item

In [ ]:
# ==========================================================================
# 3 VARIACOES DE PRE-PROCESSAMENTO
# ==========================================================================
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicação:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'análise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

# Variacao 1: Texto estruturado (padrao do winner)
def build_structured(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

# Variacao 2: Texto raw (limpeza minima, sem extracao)
def build_raw(report_text):
    return clean_text(report_text)

# Variacao 3: Texto estruturado com Achados primeiro (mais importante)
def build_achados_first(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    return ' '.join(parts)

TEXT_BUILDERS = [
    ('structured',    build_structured),
    ('raw',           build_raw),
    ('achados_first', build_achados_first),
]
print(f'TTA variacoes: {[name for name, _ in TEXT_BUILDERS]}')

In [ ]:
# ==========================================================================
# CALIBRATION & INFERENCE
# ==========================================================================
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

In [ ]:
# ==========================================================================
# CARREGAR DADOS + TOKENIZER
# ==========================================================================
test_path  = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
test_df    = pd.read_csv(test_path)
print(f'Test samples: {len(test_df)}')

tokenizer   = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')
config_path = weights_dir / 'model_config'

# Pre-carregar modelos dos 5 folds (reusar para as 3 variacoes)
fold_models = []
for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        continue
    config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    fold_models.append(model)
    del state_dict
print(f'{len(fold_models)} folds carregados na memoria.')

# ==========================================================================
# TTA: INFERENCIA COM 3 VARIACOES DE TEXTO
# ==========================================================================
tta_probs = np.zeros((len(test_df), NUM_CLASSES))

for var_name, builder_fn in TEXT_BUILDERS:
    print(f'\n--- TTA variacao: {var_name} ---')
    texts   = [builder_fn(t) for t in test_df['report'].tolist()]
    dataset = MammographyDataset(texts, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)

    var_probs = np.zeros((len(test_df), NUM_CLASSES))
    for fold_idx, model in enumerate(fold_models):
        print(f'  Fold {fold_idx}...', end=' ')
        fold_probs = predict(model, loader)
        var_probs += fold_probs
        print('ok')
    var_probs /= len(fold_models)
    tta_probs += var_probs

# Media das 3 variacoes
tta_probs /= len(TEXT_BUILDERS)
print(f'\nTTA completo: {len(TEXT_BUILDERS)} variacoes x {len(fold_models)} folds')

# Limpar memoria
for m in fold_models:
    del m
del fold_models
torch.cuda.empty_cache()

In [ ]:
# ==========================================================================
# CALIBRACAO + SUBMISSAO
# ==========================================================================
calibrated_probs = temperature_scale(tta_probs, BEST_TEMP)
predictions      = apply_thresholds(calibrated_probs, BEST_THRESHOLDS)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'submission.csv salvo ({len(submission)} linhas)')